# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset of mass spectrometry analysis using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully.")
print(f"Name: {metadata.name}\nDescription: {metadata.description}\nIdentifier: {metadata.identifier}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets and their fields using their `@id`. All subsequent operations will use these IDs for consistency.

List available record sets, their unique `@id`, and show basic information for one record from each set.

In [ ]:
# List all RecordSets in the dataset
record_sets = dataset.metadata.record_sets
print("Record Sets (@id, name):")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs['name']}")

# Preview the first record from each record set (if any):
for rs in record_sets:
    print(f"\nSample record from record set {rs['@id']} ({rs['name']}):")
    records_iterator = dataset.records(record_set=rs['@id'])
    record = next(records_iterator, None)
    if record is not None:
        pprint(record)
    else:
        print("No records found.")

## 3. Data Extraction
Load data from the main record set(s) into DataFrames for analysis. We'll use the `@id` values from the previous step. For this dataset, we choose the record set containing the protein measurements, which commonly has "protein" or "main" in its name, or otherwise the first set listed.

In [ ]:
# Define a list of all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading {rs_id}...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows and {len(df.columns)} columns for RecordSet {rs_id}.")
    else:
        print("No records found for this RecordSet.")

# For demonstration, select the first record set for exploration
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nAvailable columns in main RecordSet (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular data found to load.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All field and column references will use their `@id` values as obtained from the schema.

In [ ]:
import numpy as np

if main_record_set_id and main_record_set_id in dataframes and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    # Determine available numeric field IDs from Croissant metadata
    fields = [f for f in dataset.metadata.fields(record_set=main_record_set_id)]
    numeric_fields = [f['@id'] for f in fields if f.get('dataType', '').lower() in ['float', 'integer', 'number']]

    print("Numeric field IDs available in RecordSet:", numeric_fields)

    # Choose one numeric field for filtering (we'll use the first if present)
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Only filter if field exists and is numeric (coerce errors)
        s = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = s.dropna().quantile(0.75)  # For demonstration, use 75th percentile as threshold
        filtered_df = df[s > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (s[s > threshold] - s[s > threshold].mean()) / s[s > threshold].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a non-numeric field if available
        group_field_id = None
        for f in fields:
            if f['@id'] != numeric_field_id and f.get('dataType', '').lower() in ['text', 'string'] and f['@id'] in df.columns:
                group_field_id = f['@id']
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print('No numeric fields found for EDA.')
else:
    print('No main record set data loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll generate simple histograms and a scatter plot if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]

    # Use the same numeric_field_id as above
    if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

        # If there are at least two numeric fields, plot a scatter
        if len(numeric_fields) >= 2:
            y_field_id = numeric_fields[1]
            plt.figure(figsize=(6,6))
            sns.scatterplot(x=pd.to_numeric(df[numeric_field_id], errors='coerce'),
                            y=pd.to_numeric(df[y_field_id], errors='coerce'))
            plt.title(f"{numeric_field_id} vs {y_field_id}")
            plt.xlabel(numeric_field_id)
            plt.ylabel(y_field_id)
            plt.show()
    else:
        print('No numeric field found for visualization.')
else:
    print('No main record set data loaded for visualization.')

## 6. Conclusion
In this notebook, we have:

1. Loaded the dataset metadata and explored available record sets using the `mlcroissant` library.
2. Extracted data using `@id`-based referencing of record sets/fields as prescribed by FAIR Croissant schema.
3. Performed exploratory data analysis (EDA) by filtering, normalizing data, and grouping by attributes.
4. Visualized distributions of key numeric protein-related fields.

The FAIR² mass spectrometry dataset provides a rich resource for downstream proteomics analysis. For deeper exploration, continue with domain-specific statistical and bioinformatics methods tailored to protein quantification and annotation.